# Lab 01-03: Chain Design

| | |
|---|---|
| **Module** | 01 -- Design Applications (14% of exam) |
| **UI Sections** | Playground |
| **Estimated Time** | 30--40 minutes |
| **Difficulty** | Intermediate |

---

## What Is a Chain and Why Does It Matter?

A **chain** is a sequence of steps that process data through an AI pipeline. Think of it
like an **assembly line in a factory** -- each station does exactly one job and passes the
result to the next station. The raw material (a user question) enters at one end, gets
transformed at every stage, and a finished product (a useful answer) comes out the other end.

The exam tests whether you can **design the right chain for a given business scenario**:
pick the correct components, put them in the right order, and explain the inputs and
outputs at each step.

---

## Mental Model

> **An AI chain is like a sandwich shop order:**
>
> 1. The **Retriever** pulls ingredients from the fridge (finds relevant documents).
> 2. The **Prompt Template** assembles the sandwich (formats the context + question).
> 3. The **LLM** is the chef (generates the answer).
> 4. The **Output Parser** plates it nicely (formats the result for the user).
>
> Each step does one thing and passes to the next. If you need a different sandwich,
> you swap components -- you don't rebuild the kitchen.

---

## Exam Alert -- Common Traps

| Trap | Why Students Fall For It | The Truth |
|---|---|---|
| Putting the LLM *before* the Prompt Template | "The LLM does everything" | The LLM needs a formatted prompt first; the template builds it |
| Skipping the Retriever in a RAG chain | "The LLM already knows the answer" | Without retrieval, the LLM hallucinates or uses stale training data |
| Using a fixed chain when tools are needed | "Chains handle everything" | If the task requires dynamic decisions (e.g., call an API *or* search docs), you need an agent |
| Confusing Memory with Retrieval | "Memory = the knowledge base" | Memory stores *conversation history*; the Retriever fetches *external documents* |
| Forgetting the Output Parser | "The LLM output is fine as-is" | Raw LLM text often needs parsing into JSON, lists, or structured objects |

---

## Prerequisites

- Completed **Lab 01-01** (Application Design Fundamentals)
- Completed **Lab 01-02** (Prompt Engineering)

---

## Exam Objectives Covered

- Select chain components for a given use case
- Design an AI pipeline for a realistic business scenario
- Translate business use case goals into a description of the desired inputs and outputs for the AI pipeline
- Understand the ReAct agent pattern (Reason, Act, Observe, Repeat)

---

## Step 1: Chain Components and Their Roles

Every chain is built from a small set of reusable components. Think of them like
LEGO bricks -- each brick has a specific shape and purpose, and you snap them
together in different combinations to build different things.

| Component | What It Does | Analogy | When to Use It |
|---|---|---|---|
| **Retriever** | Searches a knowledge base and returns relevant documents | Librarian who finds the right books for your question | When the LLM needs external context (RAG) |
| **Prompt Template** | Combines the user query, retrieved context, and instructions into a formatted prompt | Recipe card that tells the chef exactly what to make | Always -- every chain needs a structured prompt |
| **LLM** | Takes the formatted prompt and generates a response | The chef who cooks the meal | Always -- this is the core reasoning engine |
| **Output Parser** | Converts raw LLM text into a structured format (JSON, list, object) | The waiter who plates the dish before serving | When downstream code needs structured data |
| **Memory** | Stores and retrieves conversation history across turns | A notebook where the waiter writes down what you ordered last time | Multi-turn conversations (chatbots) |
| **Tools** | External functions the LLM can call (APIs, databases, calculators) | Speed-dial buttons the chef can press to order special ingredients | When the LLM needs to take actions or fetch live data |

In [ ]:
# Step 1: Reference dictionary of chain components
# This is conceptual -- we are building intuition before touching LangChain (Lab 03-01).

CHAIN_COMPONENTS = {
    "Retriever": {
        "role": "Searches a knowledge base and returns relevant documents",
        "analogy": "Librarian who finds the right books",
        "input": "User query (string)",
        "output": "List of relevant document chunks",
    },
    "Prompt Template": {
        "role": "Formats the user query + context into a structured prompt",
        "analogy": "Recipe card for the chef",
        "input": "User query + retrieved documents (or other variables)",
        "output": "A single formatted string ready for the LLM",
    },
    "LLM": {
        "role": "Generates a natural-language response from a prompt",
        "analogy": "The chef who cooks the meal",
        "input": "Formatted prompt string",
        "output": "Raw text response",
    },
    "Output Parser": {
        "role": "Converts raw LLM text into structured data",
        "analogy": "Waiter who plates the dish",
        "input": "Raw LLM text",
        "output": "Structured object (dict, list, JSON, etc.)",
    },
    "Memory": {
        "role": "Stores and retrieves conversation history",
        "analogy": "Notebook tracking past orders",
        "input": "Current turn's query + response",
        "output": "Relevant past conversation context",
    },
    "Tools": {
        "role": "External functions the LLM can invoke (APIs, search, calculators)",
        "analogy": "Speed-dial buttons for special ingredients",
        "input": "Tool name + arguments (decided by the LLM)",
        "output": "Result from the external function",
    },
}

print("=" * 72)
print("CHAIN COMPONENT REFERENCE")
print("=" * 72)
for name, info in CHAIN_COMPONENTS.items():
    print(f"\n>> {name}")
    print(f"   Role    : {info['role']}")
    print(f"   Analogy : {info['analogy']}")
    print(f"   Input   : {info['input']}")
    print(f"   Output  : {info['output']}")

---

## Step 2: Design a RAG Chain

Let us design a **customer support RAG chain** from scratch. The business need:
customers ask questions about our product, and we want to answer using our
internal documentation rather than relying on the LLM's training data.

### The Chain as ASCII Art

```
User Query
    |
    v
+-------------+     +-----------------+     +-------+     +---------------+     +----------+
| Retriever   | --> | Prompt Template | --> |  LLM  | --> | Output Parser | --> | Response |
| (find docs) |     | (format prompt) |     | (gen) |     | (structure)   |     | (to user)|
+-------------+     +-----------------+     +-------+     +---------------+     +----------+
       |                    ^
       |                    |
       +--- docs -----------+
```

**Reading the diagram:**
1. The user asks a question (e.g., *"How do I reset my password?"*).
2. The **Retriever** searches the knowledge base and finds the 3 most relevant doc chunks.
3. The **Prompt Template** combines the question + docs into a single prompt string.
4. The **LLM** reads the prompt and generates an answer grounded in the docs.
5. The **Output Parser** formats the raw text (e.g., extracts just the answer, adds sources).
6. The formatted **Response** is returned to the user.

In [ ]:
# Step 2: Pseudocode walkthrough of a RAG chain
# Each function represents one component. We simulate the data flow.


def retriever(query: str) -> list:
    """Simulate searching a knowledge base."""
    # In production this would call a vector store (e.g., Databricks Vector Search).
    knowledge_base = {
        "password": "To reset your password, go to Settings > Security > Reset Password.",
        "billing":  "Billing inquiries can be handled at Settings > Billing > Contact Support.",
        "refund":   "Refunds are processed within 5-7 business days after approval.",
    }
    results = []
    for keyword, doc in knowledge_base.items():
        if keyword in query.lower():
            results.append(doc)
    # Fallback: return a generic doc if nothing matched
    if not results:
        results.append("Please contact support@example.com for further help.")
    return results


def prompt_template(query: str, docs: list) -> str:
    """Combine the query and retrieved docs into a formatted prompt."""
    context = "\n".join(f"  - {doc}" for doc in docs)
    prompt = (
        f"You are a helpful customer support assistant.\n"
        f"Answer the user's question using ONLY the context below.\n"
        f"If the context does not contain the answer, say 'I don't know'.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )
    return prompt


def llm(prompt: str) -> str:
    """Simulate an LLM generating a response."""
    # In production this calls a Foundation Model endpoint.
    # Here we just echo back a simulated answer.
    if "reset" in prompt.lower() and "password" in prompt.lower():
        return "To reset your password, navigate to Settings > Security > Reset Password."
    return "I don't have enough information to answer that. Please contact support."


def output_parser(raw_response: str) -> dict:
    """Structure the raw LLM text into a response dict."""
    return {
        "answer": raw_response.strip(),
        "confidence": "high" if "I don't" not in raw_response else "low",
    }


# ---------- RUN THE CHAIN ----------
user_query = "How do I reset my password?"

print("=" * 60)
print("RAG CHAIN EXECUTION TRACE")
print("=" * 60)

# 1. Retriever
docs = retriever(user_query)
print(f"\n[1] RETRIEVER")
print(f"    Input  : {user_query!r}")
print(f"    Output : {docs}")

# 2. Prompt Template
formatted_prompt = prompt_template(user_query, docs)
print(f"\n[2] PROMPT TEMPLATE")
print(f"    Output :\n{formatted_prompt}")

# 3. LLM
raw_answer = llm(formatted_prompt)
print(f"\n[3] LLM")
print(f"    Output : {raw_answer!r}")

# 4. Output Parser
final_response = output_parser(raw_answer)
print(f"\n[4] OUTPUT PARSER")
print(f"    Output : {final_response}")

print("\n" + "=" * 60)
print(f"FINAL ANSWER: {final_response['answer']}")
print(f"CONFIDENCE : {final_response['confidence']}")
print("=" * 60)

### Key Takeaway from Step 2

Notice how each function has a **single responsibility** and a clear input/output
contract. This modularity is what makes chains powerful:

- Want to swap the vector store? Replace the `retriever` function.
- Want a different LLM? Replace the `llm` function.
- Want JSON output? Change the `output_parser`.

The rest of the chain stays the same.

---

## Step 3: The ReAct Agent Pattern

A **chain** follows a fixed sequence -- the same steps run every time, in the same
order. But what if the LLM needs to *decide* what to do next based on intermediate
results?

That is where **agents** come in. The most common agent pattern on the exam is
**ReAct** (Reason + Act).

### The ReAct Loop

```
User Query
    |
    v
+------------------------------------------+
|           ReAct Agent Loop               |
|                                          |
|   +----------+                           |
|   | REASON   |  "I need to look up the   |
|   | (Think)  |   customer's order."       |
|   +----+-----+                           |
|        |                                 |
|        v                                 |
|   +----------+                           |
|   |   ACT    |  Call: get_order(#12345)  |
|   | (Tool)   |                           |
|   +----+-----+                           |
|        |                                 |
|        v                                 |
|   +----------+                           |
|   | OBSERVE  |  Result: "Shipped, ETA    |
|   | (Read)   |   March 20"               |
|   +----+-----+                           |
|        |                                 |
|        v                                 |
|   Enough info?                           |
|     No  --> loop back to REASON          |
|     Yes --> generate final answer        |
+------------------------------------------+
    |
    v
Final Response
```

### Chain vs. Agent -- When to Use Each

| | Chain (Fixed Sequence) | Agent (Dynamic Reasoning) |
|---|---|---|
| **Flow** | Always the same steps, same order | LLM decides the next step at each turn |
| **Tools** | Not typically used | Can call tools dynamically |
| **Predictability** | High -- easy to test and debug | Lower -- harder to predict exact path |
| **Latency** | Fast (one pass) | Slower (multiple LLM calls in a loop) |
| **Cost** | Lower (fewer LLM calls) | Higher (multiple LLM calls per query) |
| **Use when...** | The task has a fixed, known structure | The task requires decisions, branching, or tool use |
| **Example** | FAQ bot, summarizer, translator | Customer support with order lookup, research assistant |

In [ ]:
# Step 3: Simulate the ReAct agent loop


def tool_get_order(order_id: str) -> dict:
    """Simulate an order-lookup tool."""
    orders = {
        "#12345": {"status": "Shipped", "eta": "March 20", "carrier": "FedEx"},
        "#67890": {"status": "Processing", "eta": "March 25", "carrier": "UPS"},
    }
    return orders.get(order_id, {"error": "Order not found"})


def tool_get_return_policy() -> str:
    """Simulate a policy-lookup tool."""
    return "Items can be returned within 30 days. Refunds take 5-7 business days."


AVAILABLE_TOOLS = {
    "get_order": tool_get_order,
    "get_return_policy": tool_get_return_policy,
}


def react_agent(query: str, max_steps: int = 5) -> str:
    """
    Simulate a ReAct agent.

    In production the LLM decides each step. Here we hard-code
    the reasoning to demonstrate the pattern clearly.
    """
    print(f"\nUSER QUERY: {query}")
    print("-" * 50)

    # --- Simulated ReAct loop ---
    observations = []

    for step in range(1, max_steps + 1):
        # REASON: The LLM thinks about what to do next.
        if step == 1 and "order" in query.lower() and "#" in query:
            # Extract order ID from the query
            order_id = "#" + query.split("#")[1].split()[0].rstrip("?")
            thought = f"The user is asking about order {order_id}. I should look it up."
            action = "get_order"
            action_input = order_id
        elif step == 1 and "return" in query.lower():
            thought = "The user wants to know about returns. Let me check the policy."
            action = "get_return_policy"
            action_input = None
        else:
            # The agent decides it has enough info to answer.
            thought = "I have enough information to answer the question."
            print(f"\n  Step {step} - REASON: {thought}")
            print(f"           ACTION: Generate final answer")
            break

        print(f"\n  Step {step} - REASON : {thought}")
        print(f"           ACTION : Call tool '{action}'")

        # ACT: Call the tool.
        tool_fn = AVAILABLE_TOOLS[action]
        if action_input is not None:
            observation = tool_fn(action_input)
        else:
            observation = tool_fn()

        # OBSERVE: Read the result.
        print(f"           OBSERVE: {observation}")
        observations.append(observation)

    # Generate final answer from observations
    if observations and isinstance(observations[0], dict) and "status" in observations[0]:
        order_info = observations[0]
        final = (
            f"Your order is currently '{order_info['status']}'. "
            f"Estimated arrival: {order_info['eta']} via {order_info['carrier']}."
        )
    elif observations and isinstance(observations[0], str):
        final = observations[0]
    else:
        final = "I'm sorry, I couldn't find the information you need."

    print(f"\n  FINAL ANSWER: {final}")
    print("-" * 50)
    return final


# --- Run two different queries through the agent ---
print("=" * 60)
print("ReAct AGENT EXECUTION TRACES")
print("=" * 60)

react_agent("Where is my order #12345?")
print()
react_agent("What is your return policy?")

### Key Takeaway from Step 3

The ReAct agent **loops** through Reason -> Act -> Observe until it has enough
information to produce a final answer. The LLM itself decides which tool to call
and when to stop.

**Exam tip:** If a scenario mentions the need to *look up live data*, *call an API*,
or *make decisions based on intermediate results*, the answer is almost always
an **agent**, not a fixed chain.

---

## Step 4: Business Scenario to Chain Architecture Matching

On the exam you will see a business scenario and need to pick the right
architecture. This step gives you a decision framework and five practice
scenarios.

### Decision Framework

Ask these questions in order:

1. **Does the LLM need external documents?** Yes -> include a Retriever (RAG).
2. **Does the LLM need to call APIs or databases at runtime?** Yes -> use an Agent with Tools.
3. **Does the user have multi-turn conversations?** Yes -> add Memory.
4. **Is the output consumed by code (not a human)?** Yes -> add an Output Parser.
5. **Is the task a fixed sequence every time?** Yes -> Chain. No -> Agent.

In [ ]:
# Step 4: Business Scenario -> Architecture Matcher

SCENARIOS = [
    {
        "name": "Internal FAQ Bot",
        "description": (
            "Employees ask questions about company policies (PTO, benefits, "
            "IT setup). Answers come from an internal wiki."
        ),
        "needs_external_docs": True,
        "needs_live_tools": False,
        "needs_memory": False,
        "needs_structured_output": False,
    },
    {
        "name": "Customer Support with Order Lookup",
        "description": (
            "Customers ask about their orders, returns, and billing. "
            "The bot needs to look up order status in a live database."
        ),
        "needs_external_docs": True,
        "needs_live_tools": True,
        "needs_memory": True,
        "needs_structured_output": False,
    },
    {
        "name": "Document Summarizer",
        "description": (
            "Users upload a PDF and get a 3-bullet summary. "
            "No conversation, no external data."
        ),
        "needs_external_docs": False,
        "needs_live_tools": False,
        "needs_memory": False,
        "needs_structured_output": True,
    },
    {
        "name": "Research Assistant",
        "description": (
            "Analysts ask complex questions. The bot searches an internal "
            "knowledge base, calls a financial data API for live numbers, "
            "and synthesizes a report."
        ),
        "needs_external_docs": True,
        "needs_live_tools": True,
        "needs_memory": True,
        "needs_structured_output": True,
    },
    {
        "name": "Code Review Bot",
        "description": (
            "Developers paste a code snippet and get a structured review: "
            "issues found, severity, and suggested fixes. Output must be JSON."
        ),
        "needs_external_docs": False,
        "needs_live_tools": False,
        "needs_memory": False,
        "needs_structured_output": True,
    },
]


def recommend_architecture(scenario: dict) -> dict:
    """
    Given a scenario, recommend the chain architecture and list components.

    Decision logic:
    - If live tools are needed -> Agent (ReAct pattern)
    - If external docs are needed but no tools -> RAG Chain
    - Otherwise -> Simple Chain
    """
    components = ["Prompt Template", "LLM"]  # Always present

    if scenario["needs_external_docs"]:
        components.insert(0, "Retriever")

    if scenario["needs_memory"]:
        components.insert(0, "Memory")

    if scenario["needs_structured_output"]:
        components.append("Output Parser")

    if scenario["needs_live_tools"]:
        architecture = "ReAct Agent"
        components.append("Tools")
    elif scenario["needs_external_docs"]:
        architecture = "RAG Chain"
    else:
        architecture = "Simple Chain"

    return {
        "architecture": architecture,
        "components": components,
    }


# --- Run all scenarios ---
print("=" * 72)
print("BUSINESS SCENARIO -> ARCHITECTURE MATCHING")
print("=" * 72)

for s in SCENARIOS:
    result = recommend_architecture(s)
    print(f"\nScenario    : {s['name']}")
    print(f"Description : {s['description']}")
    print(f"Architecture: {result['architecture']}")
    print(f"Components  : {' -> '.join(result['components'])}")
    print("-" * 72)

### How to Read These Results on the Exam

| Scenario | Architecture | Why |
|---|---|---|
| Internal FAQ Bot | RAG Chain | Needs docs, but no live tools or branching |
| Customer Support + Orders | ReAct Agent | Must call a live order database dynamically |
| Document Summarizer | Simple Chain | Input is provided directly, no retrieval needed |
| Research Assistant | ReAct Agent | Needs both docs and live API calls, multi-step reasoning |
| Code Review Bot | Simple Chain | Input is the code snippet; output must be structured JSON |

---

## Step 4b: The Router Chain Pattern

What if your app needs to handle **multiple types of queries** with different
pipelines? For example, a GenAI assistant that answers both:
- **Document questions** (RAG chain: search docs → prompt → LLM)
- **SQL queries** (NLQ-to-SQL chain: parse intent → generate SQL → execute)

A **Router Chain** solves this by adding an **intent classification step**
at the front that routes each query to the correct specialized chain.

### The Exam Scenario (Question 60 pattern)

> *A GenAI assistant needs to support both document-based answers and
> structured SQL queries. Which design pattern fits this requirement?*
>
> - (A) Conversational memory chain -- stores history, doesn't route
> - (B) Multi-agent workflow -- overkill for two fixed paths
> - (C) RetrievalQA chain -- only handles document retrieval, not SQL
> - **(D) Router chain** -- classifies intent and routes to the right chain

```
                        User Query
                            |
                            v
                   ┌── ROUTER (LLM) ──┐
                   │  "Is this a doc   │
                   │   question or a   │
                   │   SQL query?"     │
                   └───────┬───────────┘
                      /          \\
                     v            v
              ┌─── DOC ───┐  ┌── SQL ──┐
              │ RAG chain │  │ NLQ-to  │
              │ (retrieve │  │ -SQL    │
              │  + answer)│  │ (gen +  │
              │           │  │  exec)  │
              └─────┬─────┘  └────┬────┘
                     \          /
                      v        v
                   Final Answer
```

In [ ]:
# Step 4b: Router Chain -- Intent Classification + Routing
#
# We simulate a router chain that classifies user queries and
# routes them to the correct specialized handler.

import re

# ── 1. Define the specialized chains ─────────────────────────────

def rag_chain(query):
    """Handle document-based questions via retrieval + LLM."""
    return {
        "route": "RAG",
        "action": f"Search docs for: '{query}' -> Retrieve chunks -> LLM answer",
        "simulated_answer": f"Based on our documentation, {query.lower().rstrip('?')}."
    }

def sql_chain(query):
    """Handle structured queries by generating and executing SQL."""
    return {
        "route": "SQL",
        "action": f"Parse intent -> Generate SQL -> Execute -> Format result",
        "simulated_answer": f"Query result: [structured data for '{query}']"
    }

def fallback_chain(query):
    """Handle queries that don't fit either category."""
    return {
        "route": "FALLBACK",
        "action": "Direct LLM call (no retrieval or SQL)",
        "simulated_answer": f"General response to: '{query}'"
    }

# ── 2. Build the router (intent classifier) ─────────────────────

# In production, the router is an LLM call that classifies intent.
# Here we simulate with keyword matching for demonstration.

SQL_SIGNALS = ["how many", "count", "total", "average", "sum",
               "list all", "top 10", "revenue", "sales", "group by"]
DOC_SIGNALS = ["what is", "how do", "explain", "policy", "procedure",
               "documentation", "guide", "help me understand"]

def router(query):
    """Classify intent and route to the correct chain.
    
    In production, this would be an LLM call:
        prompt = 'Classify this query as DOC or SQL: {query}'
        route = llm.invoke(prompt)  # Returns 'DOC' or 'SQL'
    """
    q_lower = query.lower()
    if any(signal in q_lower for signal in SQL_SIGNALS):
        return "SQL"
    elif any(signal in q_lower for signal in DOC_SIGNALS):
        return "DOC"
    else:
        return "FALLBACK"

ROUTES = {"DOC": rag_chain, "SQL": sql_chain, "FALLBACK": fallback_chain}

# ── 3. Test the router with different query types ────────────────

test_queries = [
    "What is the return policy for damaged items?",
    "How many orders were placed last month?",
    "Explain how model serving auto-scaling works",
    "What is the total revenue by product category?",
    "List all customers with overdue invoices",
    "Hello, how are you today?",
]

print("=" * 70)
print("ROUTER CHAIN -- Intent Classification + Routing")
print("=" * 70)

for query in test_queries:
    route = router(query)
    result = ROUTES[route](query)
    print(f"\n  Query:  '{query}'")
    print(f"  Route:  {result['route']:10s} -> {result['action']}")

print(f"\n{'=' * 70}")
print("ROUTER CHAIN vs OTHER PATTERNS")
print("=" * 70)
print("""
  Router Chain:
    - Step 1: LLM classifies intent (DOC? SQL? Chitchat?)
    - Step 2: Routes to the correct specialized chain
    - Use when: app handles MULTIPLE QUERY TYPES with different pipelines

  Why the other options are wrong:
    Conversational memory chain: Stores chat history, but doesn't ROUTE
                                 between different processing pipelines.
    Multi-agent workflow:        Multiple autonomous agents collaborating.
                                 Overkill for two fixed routes.
    RetrievalQA chain:           Only handles document retrieval (RAG).
                                 Cannot generate SQL queries.

  The router pattern is also called "multi-chain architecture" --
  one classifier at the top, specialized chains below.
""")


**Expected output:**
```
======================================================================
ROUTER CHAIN -- Intent Classification + Routing
======================================================================

  Query:  'What is the return policy for damaged items?'
  Route:  DOC  -> Search docs for: '...' -> Retrieve chunks -> LLM answer

  Query:  'How many orders were placed last month?'
  Route:  SQL  -> Parse intent -> Generate SQL -> Execute -> Format result

  Query:  'Hello, how are you today?'
  Route:  FALLBACK -> Direct LLM call (no retrieval or SQL)
```

**Key exam fact:** When the app needs to handle **both document questions AND
SQL queries**, the answer is **Router chain** (not RetrievalQA, which only does docs).

---

## Step 5: Translating Business Requirements to Pipeline I/O

This covers the exam objective:
> *"Translate business use case goals into a description of the desired inputs
> and outputs for the AI pipeline."*

### The Framework

For every business scenario, fill in this table:

| Dimension | Question to Ask |
|---|---|
| **Business Goal** | What does the business want to achieve? |
| **Input** | What does the user provide? (text, file, image, structured data?) |
| **Output** | What does the system return? (text, JSON, action, notification?) |
| **Components Needed** | Which chain components are required to bridge input to output? |
| **Quality Criteria** | How do we know the output is good? (accuracy, latency, format?) |

In [ ]:
# Step 5: Translating business requirements to pipeline I/O

BUSINESS_CASES = [
    {
        "business_goal": (
            "Reduce support ticket volume by letting customers self-serve "
            "answers about shipping, returns, and account issues."
        ),
        "input": "Natural language question (string, 1-3 sentences)",
        "output": "Natural language answer with source links (string + metadata)",
        "components": ["Retriever", "Prompt Template", "LLM", "Output Parser"],
        "quality_criteria": [
            "Answer must be grounded in retrieved docs (no hallucination)",
            "Response latency < 3 seconds",
            "Must cite the source document",
        ],
    },
    {
        "business_goal": (
            "Help sales reps generate personalized outreach emails based on "
            "a prospect's company profile and recent news."
        ),
        "input": "Company name (string) + optional talking points (list of strings)",
        "output": "Draft email (string, 150-250 words, professional tone)",
        "components": ["Retriever", "Prompt Template", "LLM"],
        "quality_criteria": [
            "Email must reference real facts from the company profile",
            "Tone must match brand guidelines",
            "Must include a clear call-to-action",
        ],
    },
    {
        "business_goal": (
            "Automatically classify incoming support tickets by urgency "
            "(critical / high / medium / low) and route to the right team."
        ),
        "input": "Support ticket text (string, 1-10 sentences)",
        "output": 'JSON object: {"urgency": "high", "team": "billing", "summary": "..."}',
        "components": ["Prompt Template", "LLM", "Output Parser"],
        "quality_criteria": [
            "Classification accuracy > 90%",
            "Output must be valid JSON (parseable by downstream systems)",
            "Latency < 2 seconds per ticket",
        ],
    },
]


print("=" * 72)
print("BUSINESS REQUIREMENT -> PIPELINE I/O MAPPING")
print("=" * 72)

for i, case in enumerate(BUSINESS_CASES, 1):
    print(f"\n{'~' * 72}")
    print(f"  CASE {i}")
    print(f"{'~' * 72}")
    print(f"  Business Goal : {case['business_goal']}")
    print(f"  Input         : {case['input']}")
    print(f"  Output        : {case['output']}")
    print(f"  Components    : {' -> '.join(case['components'])}")
    print(f"  Quality Criteria:")
    for criterion in case['quality_criteria']:
        print(f"    - {criterion}")

print(f"\n{'=' * 72}")
print("Notice the pattern: every business goal maps to a clear INPUT, a clear")
print("OUTPUT, and a specific set of COMPONENTS that bridge the two.")
print("On the exam, start by identifying the input and output -- the components")
print("will follow logically.")
print("=" * 72)

---

## Exam Tips

| # | Tip | Why It Matters |
|---|---|---|
| 0 | **Router chain for multi-type queries.** If the app handles both document questions AND SQL queries, the answer is Router chain -- it classifies intent and routes to the right sub-chain. | Distractors: RetrievalQA (docs only), conversational memory (history only) |
| 1 | **Start with Input/Output** -- before picking components, define what goes in and what comes out | This prevents you from over-engineering or picking the wrong architecture |
| 2 | **Retriever always comes before the Prompt Template** -- the template needs the docs to format the prompt | The most common ordering mistake on the exam |
| 3 | **Use agents only when needed** -- if the task is a fixed sequence, a chain is simpler, cheaper, and faster | The exam will try to tempt you with an agent when a chain is sufficient |
| 4 | **Memory is for conversation history, not knowledge** -- don't confuse it with the Retriever's document store | This distinction appears in nearly every practice test |
| 5 | **Output Parsers are required when downstream code consumes the result** -- humans can read free text; APIs cannot | If the scenario mentions JSON, structured data, or integration, you need a parser |

---

## Key Takeaways

### Chain Components
- Every chain uses at least a **Prompt Template** and an **LLM**.
- The **Retriever** adds external knowledge (RAG pattern).
- **Memory** adds conversation context across turns.
- **Output Parser** structures the response for downstream systems.
- **Tools** enable the LLM to take actions in the real world.

### Chain vs. Agent
- A **chain** is a fixed pipeline -- same steps, same order, every time.
- An **agent** uses the ReAct loop -- Reason, Act, Observe, Repeat -- to dynamically decide what to do next.
- Use a chain when the task is predictable; use an agent when it requires dynamic reasoning or tool use.

### Business Requirements to Pipeline I/O
- Always start by defining the **Input** (what the user provides) and **Output** (what the system returns).
- The components you need follow directly from the gap between input and output.
- Quality criteria (accuracy, latency, format) guide component selection and configuration.

---

## Next Lab

**Lab 01-04** builds on these chain designs by exploring **guardrails and safety** --
how to prevent the chain from producing harmful, off-topic, or incorrect outputs.
We will add input validation and output filtering as new components in the pipeline.